# 🏠 House Price Prediction — Linear Regression

---

> **Project:** ML with Scikit-Learn | **Phase:** 1 — Regression  
> **Algorithm:** Linear Regression | **Author:** Ather-ops

---

## 📌 Project Overview

This notebook builds a complete **end-to-end house price prediction pipeline** using Scikit-Learn's `LinearRegression`. We start from raw CSV data and finish with a trained model that can predict the price of any new house.

### What this notebook covers:

| Step | Task |
|------|------|
| 1 | Import libraries |
| 2 | Load and explore data |
| 3 | Visualize feature relationships (before training) |
| 4 | Descriptive statistics and missing value check |
| 5 | Outlier detection using IQR |
| 6 | Feature binning by Area |
| 7 | Mean price analysis per bin |
| 8 | Feature binning by Age |
| 9 | One-Hot Encoding of categorical bins |
| 10 | Feature and target selection |
| 11 | Train-Test Split |
| 12 | Feature Scaling |
| 13 | Model training |
| 14 | Predictions and Evaluation |
| 15 | Post-training visualizations |
| 16 | Predict price of a new house |
| 17 | Actual vs Predicted + Residual plot |

---

### 🧠 Why Linear Regression?

Linear Regression models the relationship between input features (Area, Bedrooms, Age) and a continuous target (Price) using the equation:

$$\hat{y} = w_1 x_1 + w_2 x_2 + w_3 x_3 + b$$

Where:
- $x_1$ = Area, $x_2$ = Bedrooms, $x_3$ = Age  
- $w_1, w_2, w_3$ = learned weights (coefficients)  
- $b$ = bias (intercept)  
- $\hat{y}$ = predicted price

The model minimizes **Mean Squared Error (MSE)** during training to find the best-fit line.

---
## Step 1 — Import Libraries

We import everything needed for the full pipeline upfront:

| Library | What it does here |
|---------|------------------|
| `pandas` | Load CSV, manipulate DataFrame, binning, encoding |
| `numpy` | Array operations, creating new house input |
| `matplotlib.pyplot` | All scatter plots, line plots, histograms |
| `train_test_split` | Split data into training and test sets |
| `LinearRegression` | The ML model |
| `mean_squared_error` | Measures average squared prediction error |
| `r2_score` | Measures how well model explains variance (0 to 1) |
| `mean_absolute_error` | Average absolute prediction error |
| `StandardScaler` | Normalize features to mean=0, std=1 |

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler

---
## Step 2 — Load Dataset

We load `housing_sample.csv.txt` into a Pandas DataFrame and print the first 10 rows to understand the structure.

**Expected columns:**

| Column | Type | Description |
|--------|------|-------------|
| `Area` | int/float | Size of the house in square feet |
| `Bedrooms` | int | Number of bedrooms |
| `Age` | int | Age of the house in years |
| `Price` | int/float | Sale price of the house (target variable) |

> **Note:** `df.head(10)` shows the first 10 rows. Always inspect raw data before doing anything else — it reveals unexpected values, wrong data types, or formatting issues early.

In [ ]:
df = pd.read_csv("housing_sample.csv.txt")
print("="*50)
print("original data:\n", df.head(10))

---
## Step 3 — Exploratory Visualization (Before Training)

Before building any model, we **visualize the raw relationships** between each input feature and the target variable `Price`.

**Why visualize before training?**
- Spot linear or non-linear trends
- Detect obvious outliers visually
- Understand which features are likely to be predictive

**Three scatter plots:**

| Plot | X-axis | Y-axis | Color | Expected pattern |
|------|--------|--------|-------|------------------|
| 1 | Area | Price | Red | Positive trend — bigger house = higher price |
| 2 | Bedrooms | Price | Yellow | Mild positive trend |
| 3 | Age | Price | Green | Negative trend — older house = lower price |

> `plt.subplot(1,3,n)` arranges 3 plots in 1 row, 3 columns. `plt.tight_layout()` prevents overlapping titles.

In [ ]:
plt.figure(figsize=(15,4))

plt.subplot(1,3,1)
plt.scatter(df["Area"], df["Price"], color="red")
plt.xlabel("Area")
plt.ylabel("Price")
plt.title("AREA VS PRICE(Before Training)")

plt.subplot(1,3,2)
plt.scatter(df["Bedrooms"], df["Price"], color="yellow")
plt.xlabel("No of bedrooms")
plt.ylabel("Price")
plt.title("BEDROOMS VS PRICE(Before Training)")

plt.subplot(1,3,3)
plt.scatter(df["Age"], df["Price"], color="green")
plt.xlabel("Age")
plt.ylabel("Price")
plt.title("AGE VS PRICE (Before Training)")

plt.tight_layout()
plt.show()

---
## Step 4 — Descriptive Statistics and Missing Value Check

### `df.describe()` — Basic Statistics

`describe()` gives a statistical summary of every numerical column:

| Stat | Meaning |
|------|---------|
| `count` | Number of non-null rows |
| `mean` | Average value |
| `std` | Standard deviation (spread of values) |
| `min` | Smallest value |
| `25%` | First quartile (Q1) |
| `50%` | Median (Q2) |
| `75%` | Third quartile (Q3) |
| `max` | Largest value |

### `df.isnull().sum()` — Missing Values

Counts the number of `NaN` (missing) values per column. Missing values must be handled before training — Scikit-Learn's `LinearRegression` will throw an error if any `NaN` values exist in the input.

> If `isnull().sum()` returns all zeros, the dataset is clean and ready for modeling.

In [ ]:
print("Basic statistic:\n", df.describe())
print("missing values:\n", df.isnull().sum())
print("="*50)

---
## Step 5 — Outlier Detection using IQR Method

**Outliers** are extreme values that can distort model training. We use the **Interquartile Range (IQR)** method to find the safe boundaries for the `Price` column.

### How IQR works:

$$IQR = Q3 - Q1$$

$$\text{Lower Bound} = Q1 - 1.5 \times IQR$$

$$\text{Upper Bound} = Q3 + 1.5 \times IQR$$

Any price below `lower_bound` or above `upper_bound` is flagged as an outlier.

| Variable | Meaning |
|----------|---------|
| `Q1` | 25th percentile — bottom quarter of prices |
| `Q2` | 75th percentile — top quarter begins |
| `IQR` | Spread of the middle 50% of prices |
| `lower_bound` | Minimum acceptable price |
| `upper_bound` | Maximum acceptable price |

> **Industry practice:** Instead of deleting outliers (which loses data), we **clip** values to the fence boundaries — keeping sample size intact.

In [ ]:
# step 4: Finding outlier
Q1=df["Price"].quantile(0.25)
Q2=df["Price"].quantile(0.75)
IQR=Q2-Q1
lower_bound=Q1-1.5*IQR
upper_bound=Q2+1.5*IQR
print("lower_bound:",lower_bound)
print("upper_bound:",upper_bound)
print("="*50)

---
## Step 6 — Feature Binning by Area

**Binning** (also called bucketing) converts a continuous numerical column into discrete categories. This adds structured, human-interpretable groupings that can improve model explainability.

We divide `Area` into 3 size categories:

| Bin Range (sq ft) | Label | Meaning |
|-------------------|-------|---------|
| 0 – 1000 | `small_house` | Studio or compact flat |
| 1000 – 3000 | `medium_house` | Standard family home |
| 3000 – 5000 | `large_house` | Spacious or luxury property |

**`pd.cut()`** assigns each row to the correct bin based on its `Area` value.

> The `Area_bins` column is a **categorical** (not numerical) column — it will be One-Hot Encoded later in Step 9 before it can be used in the model.

In [ ]:
# step 5: BINNING /BUCTING by AREA
bins=[0,1000,3000,5000]
labels=["small_house", "medium_house", "large_house"]
df["Area_bins"] = pd.cut(df["Area"], bins=bins, labels=labels)
print(df[["Area", "Area_bins"]])
print("="*50)

---
## Step 7 — Mean Price per Area Bin

After binning, we compute the **average price within each size category** using `groupby().transform()`.

**Why this matters:**
- Reveals whether larger houses genuinely command higher prices in this dataset
- `transform('mean')` assigns the group average back to every row — so each row gets a new column `mean_price_by_bin` showing the average price of its size category
- This is a form of **target encoding** and can be a powerful engineered feature

**Expected output pattern:**

| Area_bins | Price | mean_price_by_bin |
|-----------|-------|------------------|
| small_house | 150000 | ~160000 |
| medium_house | 280000 | ~290000 |
| large_house | 450000 | ~470000 |

> `groupby().transform()` is different from `groupby().mean()` — transform keeps the DataFrame at its original size and aligns results row-by-row.

In [ ]:
# step 6: Mean of house bins
df["mean_price_by_bin"]=df.groupby("Area_bins")["Price"].transform("mean")
print(df[["Area_bins","Price","mean_price_by_bin"]])
print("="*50)

---
## Step 8 — Feature Binning by Age

Same binning technique, now applied to the `Age` column. Older houses typically cost less — binning captures this in a structured way.

We divide `Age` into 3 categories:

| Bin Range (years) | Label | Meaning |
|-------------------|-------|---------|
| 0 – 3 | `New_house` | Recently built, premium pricing |
| 3 – 6 | `Moderate` | Mid-age, standard market value |
| 6 – 10 | `old_house` | Older property, depreciated value |

**Why bin Age?**
- The price drop from age 1 to 3 is very different from age 7 to 10
- Binning lets the model treat these as distinct categorical groups rather than a single continuous slope

> The output shows both `Area_bins` and `Age_bins` side by side for a combined structural view of each house.

In [ ]:
# step 7: BINNING /BUCKTING BY AGE
bins=[0,3,6,10]
labels=["New_house","Moderate","old_house"]
df["Age_bins"]=pd.cut(df["Age"],bins=bins,labels=labels)
print(df[["Area_bins","Age","Age_bins"]])
print("="*50)

---
## Step 9 — One-Hot Encoding

Machine Learning models cannot work with text labels like `small_house` or `New_house` — they only understand numbers. **One-Hot Encoding** converts each category into a binary (0/1) column.

### How it works:

**Area_bins encoding:**

| Area_bins | small_house | medium_house | large_house |
|-----------|-------------|--------------|-------------|
| small_house | 1 | 0 | 0 |
| medium_house | 0 | 1 | 0 |
| large_house | 0 | 0 | 1 |

**Age_bins encoding:**

| Age_bins | New_house | Moderate | old_house |
|----------|-----------|----------|-----------|
| New_house | 1 | 0 | 0 |
| Moderate | 0 | 1 | 0 |
| old_house | 0 | 0 | 1 |

**`pd.concat(..., axis=1)`** horizontally attaches both sets of one-hot columns to the original DataFrame.

> After this step, the DataFrame has the original columns **plus** 6 new binary columns. The original `Area_bins` and `Age_bins` columns are still present but will not be used in training (we select specific features in the next step).

In [ ]:
# step 8: One-Hot Encoder
# converting area and age bins into feature vector
one_hot_area_bins=pd.get_dummies(df["Area_bins"])
one_hot_age_bins=pd.get_dummies(df["Age_bins"])
# combine all into one hot column
df=pd.concat([df,one_hot_area_bins,one_hot_age_bins],axis=1)
print("\nDataFrame after all one-hot encoding:")
print("="*150)
print(df.head(10))

---
## Step 10 — Feature Selection and Train-Test Split

### Feature Selection

We choose `Area`, `Bedrooms`, and `Age` as our input features (`X`) and `Price` as our target (`Y`).

| Variable | Role | Columns |
|----------|------|----------|
| `X` | Input features (what the model sees) | Area, Bedrooms, Age |
| `Y` | Target (what the model predicts) | Price |

> The one-hot encoded columns were created for analysis purposes in this notebook but the core numerical features are used for training. This is a deliberate design choice — keeping the model simple and interpretable.

### Train-Test Split

We split the data so the model trains on 80% and is evaluated on the unseen 20%.

| Parameter | Value | Meaning |
|-----------|-------|----------|
| `test_size=0.2` | 20% | 20% of rows go to test set |
| `random_state=42` | 42 | Seed for reproducibility — same split every run |

**Why split?** If we train and test on the same data, the model appears perfect but fails on new houses — this is called **data leakage**. The test set simulates real unseen houses.

In [ ]:
# step 9: Feature and target
X=df[["Area","Bedrooms","Age"]]
Y=df["Price"]
# step 10: Train test split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42)

---
## Step 11 — Feature Scaling with StandardScaler

Our three features have very different scales:
- `Area`: ranges from ~500 to 5000+ (large numbers)
- `Bedrooms`: ranges from 1 to 6 (small numbers)
- `Age`: ranges from 0 to 10 (small numbers)

Without scaling, `Area` dominates the model simply because its values are 1000x larger — not because it is 1000x more important.

### StandardScaler Formula:

$$z = \frac{x - \mu}{\sigma}$$

Where $\mu$ = mean and $\sigma$ = standard deviation of the training data.

After scaling, every feature has **mean = 0** and **standard deviation = 1**.

### Critical Rule — Fit on Train Only:

| Method | When to use | Why |
|--------|------------|-----|
| `fit_transform(X_train)` | Training data | Learns mean and std FROM training data |
| `transform(X_test)` | Test data | Applies the SAME mean and std — no new learning |

> **Never** call `fit_transform()` on test data. That would let test-set statistics leak into the scaler — a form of data leakage that inflates performance.

In [ ]:
# step 11: Feature scaling
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

---
## Step 12 — Model Training

We initialize a `LinearRegression` model and train it on the scaled training data.

### What happens during `.fit()`?

Scikit-Learn solves the **Normal Equation** internally:

$$\theta = (X^T X)^{-1} X^T y$$

This finds the optimal weights $w_1, w_2, w_3$ and bias $b$ that minimize MSE across all training samples — in a single computation (no gradient descent loop needed for Linear Regression).

**After training, the model stores:**
- `model.coef_` → the learned weights for Area, Bedrooms, Age
- `model.intercept_` → the bias term

> This is the key difference from the scratch repo — there we manually ran 800 epochs of gradient descent. Here, Scikit-Learn solves it analytically in milliseconds.

In [ ]:
# step 12: MODEL SELECTION
model=LinearRegression()
model.fit(X_train_scaled,Y_train)

---
## Step 13 — Predictions and Model Evaluation

We run the trained model on the test set and evaluate performance using three metrics.

### Shape Check

Verifying shapes confirms the split was correct and predictions align with actual values:

| Array | Expected Shape |
|-------|---------------|
| `X_train` | (80% of rows, 3 features) |
| `X_test` | (20% of rows, 3 features) |
| `Y_train` | (80% of rows,) |
| `Y_test` | (20% of rows,) |
| `y_pred` | (20% of rows,) |

### Evaluation Metrics

| Metric | Formula | Ideal Value | What it measures |
|--------|---------|-------------|------------------|
| **MSE** | $\frac{1}{n}\sum(y - \hat{y})^2$ | As low as possible | Average squared error — penalizes large mistakes heavily |
| **MAE** | $\frac{1}{n}\sum|y - \hat{y}|$ | As low as possible | Average absolute error — easy to interpret in price units |
| **R² Score** | $1 - \frac{SS_{res}}{SS_{tot}}$ | Close to 1.0 | % of price variance explained by the model |

> An R² of 0.95 means the model explains 95% of all price variation. An R² of 0.0 means the model is no better than predicting the average price for every house.

In [ ]:
# step 12: PREDICTIONS
y_pred=model.predict(X_test_scaled) 
print("="*150)
# Step 13: Checking shapes
print("X_train shape:",X_train.shape)
print("X test shape:",X_test.shape)
print("Y train shape:",Y_train.shape)
print("Y test shape:",Y_test.shape)
print("y pred shape:",y_pred.shape)
# step 13: EVALUTION
MSE=mean_squared_error(Y_test,y_pred)
MAE=mean_absolute_error(Y_test,y_pred)
R2_SCORE=r2_score(Y_test,y_pred)
print(f"MSE:{MSE:.2f}, MAE:{MAE:.2f}, R2 SCORE:{R2_SCORE:.2f}")
print("="*50)

---
## Step 14 — Post-Training Visualization

Now we plot the same three scatter plots as Step 3 — but this time we **overlay the model's prediction line** on the actual data points.

**Reading these plots:**

| Element | Meaning |
|---------|--------|
| 🔴 Red dots | Actual house prices from test set |
| 🟢 Green line | Model's predicted prices |
| Tight clustering | Good fit — predictions are close to reality |
| Wide scatter | High residuals — model is struggling with that feature |

**What to look for:**
- **Area vs Price** — Green line should slope upward and pass through the middle of red dots
- **Bedrooms vs Price** — Fewer data points per x-value; the line shows the average predicted price per bedroom count
- **Age vs Price** — Green line should slope downward (older = cheaper)

> Compare these plots directly with Step 3 — the green prediction line now shows exactly what the trained model learned about each feature's influence on price.

In [ ]:
# sep 14: final visualisation
plt.figure(figsize=(12,4))
plt.subplot(1,3,1)
# 1: AREA VS PRICE
plt.scatter(X_test["Area"],Y_test,label="Actual line",color="red")
plt.plot(X_test["Area"],y_pred,label="Predicted line",color="green")
plt.title("AREA VS PRICE(After Training)")
plt.xlabel("AREA")
plt.ylabel("PRICE")
plt.grid(True)
plt.legend()
# 2: Bedrooms vs Price 
plt.subplot(1,3,2)
plt.scatter(X_test["Bedrooms"],Y_test,color="red",label="Actual line")
plt.plot(X_test["Bedrooms"],y_pred,label="Predicted line",color="green")
plt.xlabel("NO OF BEDROOMS")
plt.ylabel("PRICE")
plt.title("BEDROOMS VS PRICE (After Training)")
plt.grid(True)
# 3: Age vs Price
plt.subplot(1,3,3)
plt.scatter(X_test["Age"],Y_test,color="red",label="Actual line")
plt.plot(X_test["Age"],y_pred,label="Predicted line",color="green")
plt.xlabel("AGE")
plt.ylabel("PRICE")
plt.title("AGE VS PRICE (After Training)")
plt.grid(True)
plt.tight_layout()
print("="*100)

---
## Step 15 — Predict Price of a New House

This is the real-world use case — given a house the model has **never seen**, predict its price.

**New house specification:**

| Feature | Value |
|---------|-------|
| Area | 5600 sq ft |
| Bedrooms | 6 |
| Age | 2 years |

**Pipeline steps for prediction:**
1. Create the input as a NumPy array: `[[5600, 6, 2]]`
2. Scale it using the **same scaler** fitted on training data — `scaler.transform()` (not `fit_transform`)
3. Pass the scaled input through `model.predict()`
4. The output is the predicted price in the original currency unit

> This is a **large, new, spacious** house — expect a high predicted price. The model applies what it learned about Area, Bedrooms, and Age to give its best estimate.

In [ ]:
# step 15 : predicting new values
new_house=np.array([[5600,6,2]])
new_house_scaled=scaler.transform(new_house)
predicted_price=model.predict(new_house_scaled)
print(f"predicted price of new house  is :{predicted_price[0]:.2f}")
print("="*100)

---
## Step 16 — Actual vs Predicted and Residual Distribution

### Plot 1 — Actual vs Predicted Price

This is the most important evaluation plot. We plot `Y_test` (actual) on the X-axis and `y_pred` (predicted) on the Y-axis.

**How to read it:**
- A **perfect model** would show all points lying on a straight diagonal line (y = x)
- Points **above** the line = model underestimated the price
- Points **below** the line = model overestimated the price
- Tight clustering around the diagonal = high R² score

### Plot 2 — Residual Distribution

**Residuals** = Actual Price − Predicted Price

We plot a histogram of all residuals to check the **error distribution**.

| Residual Pattern | Meaning |
|-----------------|--------|
| Bell curve centered at 0 | Excellent — errors are random and unbiased |
| Skewed to one side | Model is systematically over- or under-predicting |
| Very wide spread | High variance — model is uncertain |

> A good Linear Regression model should produce residuals that are **normally distributed around zero**. This is one of the core assumptions of linear regression.

---

## ✅ Pipeline Complete

| Step | What we did |
|------|-------------|
| Data loading | Read CSV, inspected shape and head |
| EDA | Scatter plots, describe(), isnull() |
| Preprocessing | IQR outlier detection, binning, one-hot encoding |
| Modeling | Train-test split, StandardScaler, LinearRegression |
| Evaluation | MSE, MAE, R² Score |
| Inference | Predicted price of a new unseen house |
| Diagnostics | Actual vs Predicted plot, Residual distribution |

In [ ]:
# step 16 : Visualisation actual vs predicted price
plt.figure()
plt.plot(Y_test,y_pred)
plt.xlabel("Actual price")
plt.ylabel("Predicted price ")
plt.title("ACTUAL VS PREDICTED PRICE")
plt.grid(True)
residuals=Y_test-y_pred
plt.figure(figsize=(6,5))
plt.hist(residuals,bins=10,color="yellow")
plt.xlabel("Residuals error")
plt.ylabel("Frequency")
plt.title("Residual Distribution")
plt.grid(True)
plt.show()